In [ ]:
%%bash
# Cell 1: Cai moi truong RKNN + Toolchain + Export & Convert YOLOv8n-Seg/Det (320x320)
set -e
cd /content
PY=/opt/rknn_env/bin/python
MODEL_TYPE="seg"  # Chon "seg" (YOLOv8n-Seg co Masking) hoac "det" (YOLOv8n chi Box)
IMG_SIZE=320      # Chon 320 (~20-28 FPS), 416 (~16-20 FPS), hoac 640 (~10-13 FPS)

# 1. Cai Python 3.11 + rknn-toolkit2 + ultralytics (chi cai 1 lan, chay lai se bo qua)
if ! "$PY" -c "import rknn.api, ultralytics, onnx" 2>/dev/null; then
  echo "==> [1/4] Dang cai dat rknn-toolkit2 + ultralytics..."
  pip install -q uv
  [ -x "$PY" ] || uv venv --python 3.11 /opt/rknn_env
  uv pip install --python "$PY" rknn-toolkit2 "setuptools<81" "numpy<2" opencv-python ultralytics
else
  echo "==> [1/4] Moi truong /opt/rknn_env da san sang."
fi

# 2. Tai nhanh Toolchain RV1106 bang Git Sparse-Checkout + --filter=blob:none
TC_DIR="/content/luckfox-pico/tools/linux/toolchain/arm-rockchip830-linux-uclibcgnueabihf"
if [ ! -d "$TC_DIR" ]; then
  echo "==> [2/4] Dang tai Toolchain RV1106..."
  rm -rf /content/luckfox-pico
  git clone --depth 1 --filter=blob:none --sparse https://github.com/LuckfoxTECH/luckfox-pico.git /content/luckfox-pico
  git -C /content/luckfox-pico sparse-checkout set tools/linux/toolchain/arm-rockchip830-linux-uclibcgnueabihf
else
  echo "==> [2/4] Toolchain RV1106 da co san."
fi

# 3. Clone rknn_model_zoo (de lay COCO calibration + rknpu2 + jpeg_turbo)
if [ ! -d "/content/rknn_model_zoo" ]; then
  echo "==> [3/4] Dang clone rknn_model_zoo..."
  git clone --depth 1 https://github.com/airockchip/rknn_model_zoo.git /content/rknn_model_zoo
else
  echo "==> [3/4] rknn_model_zoo da co san."
fi

# 4. Tai .pt tu GitHub Ultralytics (~6.7MB), xuat ONNX chuan Rockchip va convert sang .rknn INT8
echo "==> [4/4] Dang export & convert YOLOv8n ($MODEL_TYPE, ${IMG_SIZE}x${IMG_SIZE})..."
MODEL_TYPE="$MODEL_TYPE" IMG_SIZE="$IMG_SIZE" "$PY" - << 'EOF'
import os
import types
import torch
from ultralytics import YOLO
from rknn.api import RKNN

model_type = os.environ.get("MODEL_TYPE", "seg")
img_size = int(os.environ.get("IMG_SIZE", "320"))
pt_name = "yolov8n-seg.pt" if model_type == "seg" else "yolov8n.pt"
onnx_path = f"/content/yolov8n_{model_type}_{img_size}.onnx"
dataset_txt = "/content/rknn_model_zoo/datasets/COCO/coco_subset_20.txt"

yolo = YOLO(pt_name)
m = yolo.model.eval()
head = m.model[-1]
is_seg = hasattr(head, "proto")

def rknn_head_forward(self, x):
    outs = []
    for i in range(self.nl):
        box = self.cv2[i](x[i])
        cls = torch.sigmoid(self.cv3[i](x[i]))
        score_sum = torch.clamp(cls.sum(dim=1, keepdim=True), 0.0, 1.0)
        outs.extend([box, cls, score_sum])
        if is_seg:
            outs.append(self.cv4[i](x[i]))
    if is_seg:
        outs.append(self.proto(x[0]))
    return tuple(outs)

head.forward = types.MethodType(rknn_head_forward, head)

dummy = torch.zeros(1, 3, img_size, img_size)
with torch.no_grad():
    sample_outs = m(dummy)
print(f"Exporting {pt_name} ({img_size}x{img_size}) -> {len(sample_outs)} ONNX outputs")

torch.onnx.export(
    m,
    dummy,
    onnx_path,
    opset_version=12,
    input_names=["images"],
    output_names=[f"out_{i}" for i in range(len(sample_outs))],
)

rknn = RKNN(verbose=False)
rknn.config(
    mean_values=[[0, 0, 0]],
    std_values=[[255, 255, 255]],
    target_platform="rv1106",
)
assert rknn.load_onnx(model=onnx_path) == 0
assert rknn.build(do_quantization=True, dataset=dataset_txt) == 0
assert rknn.export_rknn("/content/yolov8n.rknn") == 0
rknn.release()
print(f"Convert OK -> /content/yolov8n.rknn")
EOF

ls -lh /content/yolov8n.rknn

In [ ]:
%%bash
# Cell 2: Bien dich rknn_yolo_daemon (libjpeg-turbo NEON + YOLOv8-Seg/Det + --bin)
set -e
rm -f /content/rknn_yolo_daemon
cat << 'EOF' > /content/yolo_daemon.cc
/*
 * Compile on Google Colab (uClibc RV1106 toolchain):
 *   With libjpeg-turbo NEON (auto-detected in Cell 2):
 *     $CXX -O3 -mfpu=neon -mfloat-abi=hard -std=c++11 -DUSE_TURBOJPEG /content/yolo_daemon.cc \
 *       -I"$RKNN_INC" -I"$TJ_INC" "$TJ_LIB" -L"$RKNN_LIB" -lrknnmrt -lpthread -lm -o /content/rknn_yolo_daemon
 *   Fallback stb_image:
 *     $CXX -O3 -mfpu=neon -mfloat-abi=hard -std=c++11 /content/yolo_daemon.cc \
 *       -I"$RKNN_INC" -I"$STB_INC" -L"$RKNN_LIB" -lrknnmrt -lpthread -lm -o /content/rknn_yolo_daemon
 */
#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <string>
#include <vector>

#include "rknn_api.h"
#ifdef USE_TURBOJPEG
#include "turbojpeg.h"
#else
#define STBI_ONLY_JPEG
#define STB_IMAGE_IMPLEMENTATION
#include "stb_image.h"
#endif

struct DetBox {
  float x1, y1, x2, y2;
  float score;
  int cls_id;
  int branch;
  int grid_r;
  int grid_c;
};

struct BranchIdx {
  int box_idx = -1;
  int cls_idx = -1;
  int sum_idx = -1;
  int seg_idx = -1;
};

static const char* COCO_NAMES[80] = {
    "person","bicycle","car","motorcycle","airplane","bus","train","truck","boat","traffic light",
    "fire hydrant","stop sign","parking meter","bench","bird","cat","dog","horse","sheep","cow",
    "elephant","bear","zebra","giraffe","backpack","umbrella","handbag","tie","suitcase","frisbee",
    "skis","snowboard","sports ball","kite","baseball bat","baseball glove","skateboard","surfboard","tennis racket","bottle",
    "wine glass","cup","fork","knife","spoon","bowl","banana","apple","sandwich","orange",
    "broccoli","carrot","hot dog","pizza","donut","cake","chair","couch","potted plant","bed",
    "dining table","toilet","tv","laptop","mouse","remote","keyboard","cell phone","microwave","oven",
    "toaster","sink","refrigerator","book","clock","vase","scissors","teddy bear","hair drier","toothbrush"
};

static std::string base64_encode(const uint8_t* data, size_t len) {
  static const char* B64 =
      "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789+/";
  std::string out;
  out.reserve(((len + 2) / 3) * 4);
  for (size_t i = 0; i < len; i += 3) {
    uint32_t v = (uint32_t)data[i] << 16;
    if (i + 1 < len) v |= (uint32_t)data[i + 1] << 8;
    if (i + 2 < len) v |= (uint32_t)data[i + 2];
    out.push_back(B64[(v >> 18) & 0x3F]);
    out.push_back(B64[(v >> 12) & 0x3F]);
    out.push_back((i + 1 < len) ? B64[(v >> 6) & 0x3F] : '=');
    out.push_back((i + 2 < len) ? B64[v & 0x3F] : '=');
  }
  return out;
}

static inline float fp16_to_fp32(uint16_t h) {
  uint32_t sign = (uint32_t)(h & 0x8000) << 16;
  uint32_t exp = (h & 0x7C00) >> 10;
  uint32_t mant = (h & 0x03FF);
  uint32_t f;
  if (exp == 0) {
    if (mant == 0) {
      f = sign;
    } else {
      exp = 1;
      while ((mant & 0x0400) == 0) {
        mant <<= 1;
        exp--;
      }
      mant &= 0x03FF;
      f = sign | ((exp + 112) << 23) | (mant << 13);
    }
  } else if (exp == 31) {
    f = sign | 0x7F800000 | (mant << 13);
  } else {
    f = sign | ((exp + 112) << 23) | (mant << 13);
  }
  float res;
  memcpy(&res, &f, sizeof(res));
  return res;
}

static inline size_t elem_size_of(rknn_tensor_type t) {
  if (t == RKNN_TENSOR_FLOAT32) return 4;
  if (t == RKNN_TENSOR_FLOAT16) return 2;
  return 1;
}

struct OutView {
  int h = 0;
  int w = 0;
  int c = 0;
  int c2 = 1;
  int stride_w = 0;
  rknn_tensor_format fmt = RKNN_TENSOR_NCHW;

  void init(const rknn_tensor_attr& a) {
    fmt = a.fmt;
    size_t esz = elem_size_of(a.type);
    if (fmt == RKNN_TENSOR_NC1HWC2 && a.n_dims >= 5) {
      c2 = (int)a.dims[4];
      c = (int)a.dims[1] * c2;
      h = (int)a.dims[2];
      w = (int)a.dims[3];
      stride_w = (a.w_stride >= (uint32_t)w) ? (int)a.w_stride : w;
    } else if (fmt == RKNN_TENSOR_NHWC && a.n_dims >= 4) {
      h = (int)a.dims[1];
      w = (int)a.dims[2];
      c = (int)a.dims[3];
      if (a.w_stride > (uint32_t)w &&
          (size_t)h * a.w_stride * c * esz <= a.size_with_stride &&
          a.size_with_stride > (size_t)h * w * c * esz) {
        stride_w = (int)a.w_stride;
      } else {
        stride_w = w;
      }
    } else {
      c = (int)a.dims[1];
      h = (int)a.dims[2];
      w = (int)a.dims[3];
      stride_w = (a.w_stride > (uint32_t)w &&
                  (size_t)c * h * a.w_stride * esz <= a.size_with_stride)
                     ? (int)a.w_stride
                     : w;
    }
  }

  inline int offset(int r, int col, int ch) const {
    if (fmt == RKNN_TENSOR_NHWC) {
      return (r * stride_w + col) * c + ch;
    }
    if (fmt == RKNN_TENSOR_NC1HWC2) {
      int c1_idx = ch / c2;
      int c2_idx = ch % c2;
      return (((c1_idx * h + r) * stride_w) + col) * c2 + c2_idx;
    }
    return (ch * h + r) * stride_w + col;
  }
};

static inline float get_val(const void* data, int idx, const rknn_tensor_attr& attr) {
  if (attr.type == RKNN_TENSOR_INT8)
    return ((float)((const int8_t*)data)[idx] - (float)attr.zp) * attr.scale;
  if (attr.type == RKNN_TENSOR_UINT8)
    return ((float)((const uint8_t*)data)[idx] - (float)attr.zp) * attr.scale;
  if (attr.type == RKNN_TENSOR_FLOAT16)
    return fp16_to_fp32(((const uint16_t*)data)[idx]);
  return ((const float*)data)[idx];
}

static inline int32_t quant_thresh_i8(float thresh, const rknn_tensor_attr& attr) {
  if (attr.scale <= 0.0f) return -128;
  float q = floorf(thresh / attr.scale + (float)attr.zp);
  if (q < -128.0f) return -128;
  if (q > 127.0f) return 127;
  return (int32_t)q;
}

static float iou(const DetBox& a, const DetBox& b) {
  float xx1 = std::max(a.x1, b.x1), yy1 = std::max(a.y1, b.y1);
  float xx2 = std::min(a.x2, b.x2), yy2 = std::min(a.y2, b.y2);
  float inter = std::max(0.0f, xx2 - xx1) * std::max(0.0f, yy2 - yy1);
  float area_a = std::max(0.0f, a.x2 - a.x1) * std::max(0.0f, a.y2 - a.y1);
  float area_b = std::max(0.0f, b.x2 - b.x1) * std::max(0.0f, b.y2 - b.y1);
  return inter / (area_a + area_b - inter + 1e-6f);
}

static void letterbox_rgb(const uint8_t* src, int sw, int sh,
                          uint8_t* dst, int dw, int dh, int dst_stride,
                          float& scale, int& pad_x, int& pad_y) {
  scale = std::min((float)dw / sw, (float)dh / sh);
  int nw = (int)(sw * scale), nh = (int)(sh * scale);
  pad_x = (dw - nw) / 2;
  pad_y = (dh - nh) / 2;
  memset(dst, 114, (size_t)dst_stride * dh * 3);

  if (nw == sw && nh == sh) {
    size_t row_bytes = (size_t)sw * 3;
    for (int y = 0; y < sh; y++) {
      memcpy(dst + ((y + pad_y) * dst_stride + pad_x) * 3, src + y * row_bytes, row_bytes);
    }
    return;
  }

  std::vector<int> x_lut(nw);
  std::vector<int> y_lut(nh);
  for (int x = 0; x < nw; x++) x_lut[x] = std::min(sw - 1, (int)(x / scale));
  for (int y = 0; y < nh; y++) y_lut[y] = std::min(sh - 1, (int)(y / scale));

  for (int y = 0; y < nh; y++) {
    uint8_t* drow = dst + ((y + pad_y) * dst_stride + pad_x) * 3;
    const uint8_t* srow = src + y_lut[y] * sw * 3;
    for (int x = 0; x < nw; x++) {
      const uint8_t* sp = srow + x_lut[x] * 3;
      drow[x * 3 + 0] = sp[0];
      drow[x * 3 + 1] = sp[1];
      drow[x * 3 + 2] = sp[2];
    }
  }
}

static void decode_branch_rv1106(
    int branch_id,
    const void* box_buf, const rknn_tensor_attr& box_attr, const OutView& box_view,
    const void* cls_buf, const rknn_tensor_attr& cls_attr, const OutView& cls_view,
    const void* sum_buf, const rknn_tensor_attr* sum_attr, const OutView* sum_view,
    int model_h, int sw, int sh, float scale, int pad_x, int pad_y,
    float conf_thresh, std::vector<DetBox>& out) {
  int grid_h = box_view.h;
  int grid_w = box_view.w;
  if (grid_h <= 0 || grid_w <= 0) return;

  int stride = model_h / grid_h;
  int dfl_len = box_view.c / 4;
  if (dfl_len <= 0 || dfl_len > 32) return;
  int num_cls = cls_view.c;

  bool sum_i8 = (sum_buf && sum_attr && sum_attr->type == RKNN_TENSOR_INT8);
  bool cls_i8 = (cls_attr.type == RKNN_TENSOR_INT8);
  int32_t sum_q_th = sum_i8 ? quant_thresh_i8(conf_thresh, *sum_attr) : -128;
  int32_t cls_q_th = cls_i8 ? quant_thresh_i8(conf_thresh, cls_attr) : -128;

  const int8_t* sum_p = (const int8_t*)sum_buf;
  const int8_t* cls_p = (const int8_t*)cls_buf;

  for (int r = 0; r < grid_h; r++) {
    for (int c = 0; c < grid_w; c++) {
      if (sum_buf && sum_attr && sum_view) {
        int soff = sum_view->offset(r, c, 0);
        if (sum_i8) {
          if (sum_p[soff] < sum_q_th) continue;
        } else {
          if (get_val(sum_buf, soff, *sum_attr) < conf_thresh) continue;
        }
      }

      int best_cls = -1;
      float best_score = 0.0f;
      if (cls_i8) {
        int32_t max_q = cls_q_th - 1;
        if (cls_view.fmt == RKNN_TENSOR_NHWC) {
          const int8_t* cell_cls = cls_p + (r * cls_view.stride_w + c) * num_cls;
          for (int k = 0; k < num_cls; k++) {
            int32_t q = cell_cls[k];
            if (q > max_q) {
              max_q = q;
              best_cls = k;
            }
          }
        } else {
          for (int k = 0; k < num_cls; k++) {
            int32_t q = cls_p[cls_view.offset(r, c, k)];
            if (q > max_q) {
              max_q = q;
              best_cls = k;
            }
          }
        }
        if (best_cls < 0) continue;
        best_score = ((float)max_q - (float)cls_attr.zp) * cls_attr.scale;
      } else {
        best_score = conf_thresh;
        for (int k = 0; k < num_cls; k++) {
          float v = get_val(cls_buf, cls_view.offset(r, c, k), cls_attr);
          if (v >= best_score) {
            best_score = v;
            best_cls = k;
          }
        }
        if (best_cls < 0) continue;
      }

      if (best_score < conf_thresh) continue;

      float ltrb[4] = {0.0f, 0.0f, 0.0f, 0.0f};
      for (int b = 0; b < 4; b++) {
        float probs[32];
        float max_v = -1e9f;
        int base = b * dfl_len;
        for (int d = 0; d < dfl_len; d++) {
          float v = get_val(box_buf, box_view.offset(r, c, base + d), box_attr);
          probs[d] = v;
          if (v > max_v) max_v = v;
        }
        float sum_e = 0.0f, acc = 0.0f;
        for (int d = 0; d < dfl_len; d++) {
          float e = expf(probs[d] - max_v);
          probs[d] = e;
          sum_e += e;
        }
        for (int d = 0; d < dfl_len; d++) {
          acc += (probs[d] / sum_e) * d;
        }
        ltrb[b] = acc * stride;
      }

      float cx = (c + 0.5f) * stride;
      float cy = (r + 0.5f) * stride;
      float x1 = std::max(0.0f, std::min((float)sw, ((cx - ltrb[0]) - pad_x) / scale));
      float y1 = std::max(0.0f, std::min((float)sh, ((cy - ltrb[1]) - pad_y) / scale));
      float x2 = std::max(0.0f, std::min((float)sw, ((cx + ltrb[2]) - pad_x) / scale));
      float y2 = std::max(0.0f, std::min((float)sh, ((cy + ltrb[3]) - pad_y) / scale));
      if (x2 <= x1 || y2 <= y1) continue;
      DetBox db;
      db.x1 = x1;
      db.y1 = y1;
      db.x2 = x2;
      db.y2 = y2;
      db.score = best_score;
      db.cls_id = best_cls;
      db.branch = branch_id;
      db.grid_r = r;
      db.grid_c = c;
      out.push_back(db);
    }
  }
}

struct MaskOut {
  int w = 0;
  int h = 0;
  float rx1 = 0, ry1 = 0, rx2 = 0, ry2 = 0;
  std::string b64;
};

static bool build_mask_b64(
    const DetBox& d,
    const void* seg_buf, const rknn_tensor_attr& seg_attr, const OutView& seg_view,
    const void* proto_buf, const rknn_tensor_attr& proto_attr, const OutView& proto_view,
    int model_w, int model_h, float scale, int pad_x, int pad_y,
    MaskOut& out) {
  int nc = std::min(32, proto_view.c);
  if (nc <= 0 || proto_view.w <= 0 || proto_view.h <= 0) return false;

  float mc[32];
  float mc_sum = 0.0f;
  for (int k = 0; k < nc; k++) {
    mc[k] = get_val(seg_buf, seg_view.offset(d.grid_r, d.grid_c, k), seg_attr);
    mc_sum += mc[k];
  }

  float proto_sx = (float)proto_view.w / (float)model_w;
  float proto_sy = (float)proto_view.h / (float)model_h;
  float mx1 = d.x1 * scale + (float)pad_x;
  float my1 = d.y1 * scale + (float)pad_y;
  float mx2 = d.x2 * scale + (float)pad_x;
  float my2 = d.y2 * scale + (float)pad_y;

  int px1 = std::max(0, (int)floorf(mx1 * proto_sx));
  int py1 = std::max(0, (int)floorf(my1 * proto_sy));
  int px2 = std::min(proto_view.w, (int)ceilf(mx2 * proto_sx));
  int py2 = std::min(proto_view.h, (int)ceilf(my2 * proto_sy));
  int mw = px2 - px1;
  int mh = py2 - py1;
  if (mw <= 0 || mh <= 0) return false;

  out.w = mw;
  out.h = mh;
  out.rx1 = ((float)px1 / proto_sx - (float)pad_x) / scale;
  out.ry1 = ((float)py1 / proto_sy - (float)pad_y) / scale;
  out.rx2 = ((float)px2 / proto_sx - (float)pad_x) / scale;
  out.ry2 = ((float)py2 / proto_sy - (float)pad_y) / scale;

  std::vector<uint8_t> bits(((size_t)mw * mh + 7) / 8, 0);
  bool proto_i8 = (proto_attr.type == RKNN_TENSOR_INT8 && proto_attr.scale > 0.0f);
  const int8_t* p_i8 = (const int8_t*)proto_buf;
  float zp_mc_sum = (float)proto_attr.zp * mc_sum;

  size_t bit_idx = 0;
  for (int py = py1; py < py2; py++) {
    for (int px = px1; px < px2; px++, bit_idx++) {
      bool on = false;
      if (proto_i8) {
        float acc = 0.0f;
        if (proto_view.fmt == RKNN_TENSOR_NHWC) {
          const int8_t* cell = p_i8 + (py * proto_view.stride_w + px) * proto_view.c;
          for (int k = 0; k < nc; k++) acc += mc[k] * (float)cell[k];
        } else {
          for (int k = 0; k < nc; k++) {
            acc += mc[k] * (float)p_i8[proto_view.offset(py, px, k)];
          }
        }
        on = (acc > zp_mc_sum);
      } else {
        float acc = 0.0f;
        for (int k = 0; k < nc; k++) {
          acc += mc[k] * get_val(proto_buf, proto_view.offset(py, px, k), proto_attr);
        }
        on = (acc > 0.0f);
      }
      if (on) {
        bits[bit_idx >> 3] |= (uint8_t)(1u << (bit_idx & 7));
      }
    }
  }

  out.b64 = base64_encode(bits.data(), bits.size());
  return true;
}

static bool read_exact(FILE* fp, void* dst, size_t n) {
  uint8_t* p = (uint8_t*)dst;
  size_t got = 0;
  while (got < n) {
    size_t r = fread(p + got, 1, n - got, fp);
    if (r == 0) return false;
    got += r;
  }
  return true;
}

int main(int argc, char** argv) {
  if (argc < 2) {
    fprintf(stderr, "Usage: %s <yolov8n.rknn> [--bin]\n", argv[0]);
    return 1;
  }
  bool bin_mode = (argc >= 3 && strcmp(argv[2], "--bin") == 0);

  FILE* fp = fopen(argv[1], "rb");
  if (!fp) {
    fprintf(stderr, "Cannot open model %s\n", argv[1]);
    return 1;
  }
  fseek(fp, 0, SEEK_END);
  size_t sz = ftell(fp);
  fseek(fp, 0, SEEK_SET);
  std::vector<uint8_t> model_buf(sz);
  if (fread(model_buf.data(), 1, sz, fp) != sz) {
    fclose(fp);
    return 1;
  }
  fclose(fp);

  rknn_context ctx = 0;
  int ret = rknn_init(&ctx, model_buf.data(), sz, 0, nullptr);
  if (ret < 0) {
    fprintf(stderr, "rknn_init fail %d\n", ret);
    return 1;
  }

  rknn_input_output_num io_num{};
  rknn_query(ctx, RKNN_QUERY_IN_OUT_NUM, &io_num, sizeof(io_num));

  rknn_tensor_attr in_attr{};
  in_attr.index = 0;
  rknn_query(ctx, RKNN_QUERY_NATIVE_INPUT_ATTR, &in_attr, sizeof(in_attr));
  in_attr.type = RKNN_TENSOR_UINT8;
  in_attr.fmt = RKNN_TENSOR_NHWC;
  int model_h = (int)in_attr.dims[1];
  int model_w = (int)in_attr.dims[2];
  int in_stride = (in_attr.w_stride >= (uint32_t)model_w) ? (int)in_attr.w_stride : model_w;

  rknn_tensor_mem* in_mem = rknn_create_mem(ctx, in_attr.size_with_stride);
  if (!in_mem || rknn_set_io_mem(ctx, in_mem, &in_attr) < 0) {
    fprintf(stderr, "rknn_set_io_mem in fail\n");
    return 1;
  }

  std::vector<rknn_tensor_attr> out_attrs(io_num.n_output);
  std::vector<rknn_tensor_mem*> out_mems(io_num.n_output, nullptr);
  std::vector<OutView> out_views(io_num.n_output);
  for (uint32_t i = 0; i < io_num.n_output; i++) {
    memset(&out_attrs[i], 0, sizeof(rknn_tensor_attr));
    out_attrs[i].index = i;
    ret = rknn_query(ctx, RKNN_QUERY_NATIVE_NHWC_OUTPUT_ATTR, &out_attrs[i], sizeof(rknn_tensor_attr));
    if (ret < 0) {
      memset(&out_attrs[i], 0, sizeof(rknn_tensor_attr));
      out_attrs[i].index = i;
      ret = rknn_query(ctx, RKNN_QUERY_NATIVE_OUTPUT_ATTR, &out_attrs[i], sizeof(rknn_tensor_attr));
      if (ret < 0) {
        fprintf(stderr, "rknn_query out[%u] fail %d\n", i, ret);
        return 1;
      }
    }
    out_mems[i] = rknn_create_mem(ctx, out_attrs[i].size_with_stride);
    if (!out_mems[i] || rknn_set_io_mem(ctx, out_mems[i], &out_attrs[i]) < 0) {
      fprintf(stderr, "rknn_set_io_mem out[%u] fail\n", i);
      return 1;
    }
    out_views[i].init(out_attrs[i]);
    fprintf(stderr,
            "out[%u] fmt=%d dims=[%u,%u,%u,%u,%u] w_stride=%u type=%d zp=%d scale=%.6f sz=%u sws=%u\n",
            i, out_attrs[i].fmt, out_attrs[i].dims[0], out_attrs[i].dims[1],
            out_attrs[i].dims[2], out_attrs[i].dims[3], out_attrs[i].dims[4],
            out_attrs[i].w_stride, out_attrs[i].type, out_attrs[i].zp,
            out_attrs[i].scale, out_attrs[i].size, out_attrs[i].size_with_stride);
  }

  int proto_idx = -1;
  for (uint32_t i = 0; i < io_num.n_output; i++) {
    if (out_views[i].h == model_h / 4 && out_views[i].w == model_w / 4 && out_views[i].c == 32) {
      proto_idx = (int)i;
      break;
    }
  }
  bool has_seg = (proto_idx >= 0);

  BranchIdx branches[3];
  const int strides[3] = {8, 16, 32};
  for (uint32_t i = 0; i < io_num.n_output; i++) {
    if ((int)i == proto_idx) continue;
    int b = -1;
    for (int k = 0; k < 3; k++) {
      if (out_views[i].h == model_h / strides[k] && out_views[i].w == model_w / strides[k]) {
        b = k;
        break;
      }
    }
    if (b < 0) continue;
    if (out_views[i].c == 64 && branches[b].box_idx < 0) {
      branches[b].box_idx = (int)i;
    } else if (out_views[i].c == 1 && branches[b].sum_idx < 0) {
      branches[b].sum_idx = (int)i;
    } else if (has_seg && out_views[i].c == 32 && branches[b].seg_idx < 0) {
      branches[b].seg_idx = (int)i;
    } else if (branches[b].cls_idx < 0) {
      branches[b].cls_idx = (int)i;
    }
  }
  fprintf(stderr, "YOLOv8 mode: %s (proto=%d, b0=[%d,%d,%d,%d])\n",
          has_seg ? "SEG" : "DET", proto_idx,
          branches[0].box_idx, branches[0].cls_idx, branches[0].sum_idx, branches[0].seg_idx);

#ifdef USE_TURBOJPEG
  tjhandle tj = tjInitDecompress();
  if (!tj) {
    fprintf(stderr, "tjInitDecompress fail\n");
    return 1;
  }
  const char* decoder_name = "turbo";
  std::vector<uint8_t> rgb_buf;
#else
  const char* decoder_name = "stb";
#endif

  printf("%s\n", bin_mode ? "READY_BIN" : "READY");
  fflush(stdout);

  const float conf_thresh = 0.20f;
  const float nms_thresh = 0.45f;
  std::vector<uint8_t> jpg_buf;
  char line[512];

  while (true) {
    int sw = 0, sh = 0;
    uint8_t* img = nullptr;
    uint32_t frame_sz = 0;

    auto td0 = std::chrono::steady_clock::now();
    if (bin_mode) {
      uint8_t hdr[4];
      if (!read_exact(stdin, hdr, 4)) break;
      frame_sz = ((uint32_t)hdr[0] << 24) | ((uint32_t)hdr[1] << 16) |
                 ((uint32_t)hdr[2] << 8) | (uint32_t)hdr[3];
      if (frame_sz == 0 || frame_sz > 8u * 1024u * 1024u) {
        printf("{\"err\":\"bad frame size\",\"dets\":[]}\n");
        fflush(stdout);
        break;
      }
      if (jpg_buf.size() < frame_sz) jpg_buf.resize(frame_sz);
      if (!read_exact(stdin, jpg_buf.data(), frame_sz)) break;
    } else {
      if (!fgets(line, sizeof(line), stdin)) break;
      size_t len = strlen(line);
      while (len > 0 && (line[len - 1] == '\n' || line[len - 1] == '\r')) line[--len] = '\0';
      if (len == 0) continue;
#ifdef USE_TURBOJPEG
      FILE* jf = fopen(line, "rb");
      if (jf) {
        fseek(jf, 0, SEEK_END);
        long fsz = ftell(jf);
        fseek(jf, 0, SEEK_SET);
        if (fsz > 0 && fsz <= 8L * 1024L * 1024L) {
          frame_sz = (uint32_t)fsz;
          if (jpg_buf.size() < frame_sz) jpg_buf.resize(frame_sz);
          if (fread(jpg_buf.data(), 1, frame_sz, jf) != frame_sz) frame_sz = 0;
        }
        fclose(jf);
      }
#endif
    }

#ifdef USE_TURBOJPEG
    if (frame_sz > 0) {
      int subsamp = 0, cs = 0;
      if (tjDecompressHeader3(tj, jpg_buf.data(), (unsigned long)frame_sz,
                              &sw, &sh, &subsamp, &cs) == 0 &&
          sw > 0 && sh > 0) {
        size_t need = (size_t)sw * (size_t)sh * 3;
        if (rgb_buf.size() < need) rgb_buf.resize(need);
        if (tjDecompress2(tj, jpg_buf.data(), (unsigned long)frame_sz,
                          rgb_buf.data(), sw, 0, sh, TJPF_RGB, TJFLAG_FASTDCT) == 0) {
          img = rgb_buf.data();
        }
      }
    }
#else
    int sc = 0;
    if (bin_mode) {
      img = stbi_load_from_memory(jpg_buf.data(), (int)frame_sz, &sw, &sh, &sc, 3);
    } else {
      img = stbi_load(line, &sw, &sh, &sc, 3);
    }
#endif
    auto td1 = std::chrono::steady_clock::now();
    double dec_ms = std::chrono::duration<double, std::milli>(td1 - td0).count();

    if (!img) {
      printf("{\"err\":\"decode fail\",\"decoder\":\"%s\",\"dec_ms\":%.2f,\"dets\":[]}\n",
             decoder_name, dec_ms);
      fflush(stdout);
      continue;
    }

    float scale = 1.0f;
    int pad_x = 0, pad_y = 0;
    letterbox_rgb(img, sw, sh, (uint8_t*)in_mem->virt_addr,
                  model_w, model_h, in_stride, scale, pad_x, pad_y);
#ifndef USE_TURBOJPEG
    stbi_image_free(img);
#endif

    auto t0 = std::chrono::steady_clock::now();
    rknn_mem_sync(ctx, in_mem, RKNN_MEMORY_SYNC_TO_DEVICE);
    ret = rknn_run(ctx, nullptr);
    for (uint32_t i = 0; i < io_num.n_output; i++) {
      rknn_mem_sync(ctx, out_mems[i], RKNN_MEMORY_SYNC_FROM_DEVICE);
    }
    auto t1 = std::chrono::steady_clock::now();
    double npu_ms = std::chrono::duration<double, std::milli>(t1 - t0).count();

    auto tp0 = std::chrono::steady_clock::now();
    std::vector<DetBox> cand;
    if (ret >= 0) {
      for (int b = 0; b < 3; b++) {
        int box_idx = branches[b].box_idx;
        int cls_idx = branches[b].cls_idx;
        int sum_idx = branches[b].sum_idx;
        if (box_idx < 0 || cls_idx < 0) continue;
        decode_branch_rv1106(
            b,
            out_mems[box_idx]->virt_addr, out_attrs[box_idx], out_views[box_idx],
            out_mems[cls_idx]->virt_addr, out_attrs[cls_idx], out_views[cls_idx],
            sum_idx >= 0 ? out_mems[sum_idx]->virt_addr : nullptr,
            sum_idx >= 0 ? &out_attrs[sum_idx] : nullptr,
            sum_idx >= 0 ? &out_views[sum_idx] : nullptr,
            model_h, sw, sh, scale, pad_x, pad_y, conf_thresh, cand);
      }
    }

    std::sort(cand.begin(), cand.end(),
              [](const DetBox& a, const DetBox& b) { return a.score > b.score; });
    std::vector<DetBox> kept;
    for (const auto& c : cand) {
      bool sup = false;
      for (const auto& k : kept) {
        if (k.cls_id == c.cls_id && iou(c, k) > nms_thresh) {
          sup = true;
          break;
        }
      }
      if (!sup) kept.push_back(c);
      if (kept.size() >= 25) break;
    }

    std::vector<MaskOut> masks(kept.size());
    std::vector<bool> mask_ok(kept.size(), false);
    if (has_seg) {
      size_t limit = std::min<size_t>(kept.size(), 10);
      for (size_t i = 0; i < limit; i++) {
        const auto& d = kept[i];
        int seg_idx = (d.branch >= 0 && d.branch < 3) ? branches[d.branch].seg_idx : -1;
        if (seg_idx < 0) continue;
        mask_ok[i] = build_mask_b64(
            d,
            out_mems[seg_idx]->virt_addr, out_attrs[seg_idx], out_views[seg_idx],
            out_mems[proto_idx]->virt_addr, out_attrs[proto_idx], out_views[proto_idx],
            model_w, model_h, scale, pad_x, pad_y, masks[i]);
      }
    }
    auto tp1 = std::chrono::steady_clock::now();
    double post_ms = std::chrono::duration<double, std::milli>(tp1 - tp0).count();

    printf("{\"npu_ms\":%.2f,\"dec_ms\":%.2f,\"post_ms\":%.2f,\"decoder\":\"%s\","
           "\"has_seg\":%s,\"model_w\":%d,\"model_h\":%d,\"w\":%d,\"h\":%d,\"dets\":[",
           npu_ms, dec_ms, post_ms, decoder_name,
           has_seg ? "true" : "false", model_w, model_h, sw, sh);
    for (size_t i = 0; i < kept.size(); i++) {
      const auto& d = kept[i];
      const char* lbl = (d.cls_id >= 0 && d.cls_id < 80) ? COCO_NAMES[d.cls_id] : "obj";
      if (i > 0) printf(",");
      printf("{\"label\":\"%s\",\"cls\":%d,\"score\":%.3f,\"box\":[%.1f,%.1f,%.1f,%.1f]",
             lbl, d.cls_id, d.score, d.x1, d.y1, d.x2, d.y2);
      if (mask_ok[i]) {
        const auto& m = masks[i];
        printf(",\"mask\":{\"w\":%d,\"h\":%d,\"rect\":[%.1f,%.1f,%.1f,%.1f],\"b64\":\"%s\"}",
               m.w, m.h, m.rx1, m.ry1, m.rx2, m.ry2, m.b64.c_str());
      }
      printf("}");
    }
    printf("]}\n");
    fflush(stdout);
  }

#ifdef USE_TURBOJPEG
  tjDestroy(tj);
#endif
  rknn_destroy_mem(ctx, in_mem);
  for (auto* m : out_mems) rknn_destroy_mem(ctx, m);
  rknn_destroy(ctx);
  return 0;
}
EOF

CXX=/content/luckfox-pico/tools/linux/toolchain/arm-rockchip830-linux-uclibcgnueabihf/bin/arm-rockchip830-linux-uclibcgnueabihf-g++
RKNN_INC=/content/rknn_model_zoo/3rdparty/rknpu2/include
RKNN_LIB=/content/rknn_model_zoo/3rdparty/rknpu2/Linux/armhf-uclibc
STB_INC=/content/rknn_model_zoo/3rdparty/stb_image

TJ_LIB=""
for p in \
  /content/rknn_model_zoo/3rdparty/jpeg_turbo/Linux/armhf_uclibc/libturbojpeg.a \
  /content/rknn_model_zoo/3rdparty/jpeg_turbo/Linux/armhf-uclibc/libturbojpeg.a; do
  if [ -f "$p" ]; then TJ_LIB="$p"; break; fi
done
if [ -z "$TJ_LIB" ] && [ -d /content/rknn_model_zoo/3rdparty/jpeg_turbo ]; then
  TJ_LIB="$(find /content/rknn_model_zoo/3rdparty/jpeg_turbo -name 'libturbojpeg.a' | grep -i uclibc | head -n 1 || true)"
fi
TJ_HDR="$(find /content/rknn_model_zoo/3rdparty/jpeg_turbo -name 'turbojpeg.h' 2>/dev/null | head -n 1 || true)"
TJ_INC=""
if [ -n "$TJ_HDR" ]; then TJ_INC="$(dirname "$TJ_HDR")"; fi

BUILT=0
if [ -n "$TJ_LIB" ] && [ -n "$TJ_INC" ]; then
  echo "==> Thu bien dich voi libjpeg-turbo NEON: $TJ_LIB"
  if "$CXX" -O3 -mfpu=neon -mfloat-abi=hard -std=c++11 -DUSE_TURBOJPEG /content/yolo_daemon.cc \
      -I"$RKNN_INC" -I"$TJ_INC" "$TJ_LIB" \
      -L"$RKNN_LIB" -lrknnmrt -lpthread -lm \
      -o /content/rknn_yolo_daemon; then
    BUILT=1
    echo "==> Build OK voi decoder: libjpeg-turbo (NEON)"
  else
    echo "==> Link libjpeg-turbo loi, chuyen sang stb_image fallback..."
  fi
fi

if [ "$BUILT" -eq 0 ]; then
  "$CXX" -O3 -mfpu=neon -mfloat-abi=hard -std=c++11 /content/yolo_daemon.cc \
    -I"$RKNN_INC" -I"$STB_INC" \
    -L"$RKNN_LIB" -lrknnmrt -lpthread -lm \
    -o /content/rknn_yolo_daemon
  echo "==> Build OK voi decoder: stb_image"
fi

grep -qa "has_seg" /content/rknn_yolo_daemon && echo "==> VERIFIED: binary ho tro YOLOv8-Seg + Masking!"
ls -lh /content/rknn_yolo_daemon

### Tải bộ cài đặt và Model về máy tính cá nhân

Chạy ô code dưới đây để đóng gói file chạy `rknn_yolo_daemon` cùng mô hình `yolov8n.rknn` thành một file zip duy nhất và tải về máy.

In [11]:
import zipfile
from google.colab import files
import os

# Khởi tạo danh sách các file cần nén
files_to_download = [
    '/content/yolov8n.rknn',
    '/content/rknn_yolo_daemon'
]

zip_filename = '/content/luckfox_yolov8_deploy.zip'

# Tạo file nén zip
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file in files_to_download:
        if os.path.exists(file):
            zipf.write(file, os.path.basename(file))
            print(f"Đã thêm: {os.path.basename(file)}")
        else:
            print(f"Lỗi: Không tìm thấy file {file}")

print("Đang chuẩn bị tải xuống file zip...")
files.download(zip_filename)

Đã thêm: yolov8n.rknn
Đã thêm: rknn_yolo_daemon
Đang chuẩn bị tải xuống file zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>